In [1]:
%pwd

'a:\\projects\\text-summarizer\\research'

In [2]:
import os
os.chdir('../')
%pwd

'a:\\projects\\text-summarizer'

# Entity

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

# Configuration manager

In [4]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories
from pathlib import Path

In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories(path_to_directories=[Path(self.config.artifacts_root)])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories(path_to_directories=[Path(config.root_dir)])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_path=Path(config.model_path),
            tokenizer_path=Path(config.tokenizer_path),
            metric_file_name=Path(config.metric_file_name)
        )

        return model_evaluation_config

# Components

In [9]:
pip install evaluate rouge_score absl-py

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import evaluate 
from textSummarizer.logging import logger

In [13]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split the dataset into smaller batches that we can process simultaneously.
        Yield successive batch-sized chunks from list_of_elements.
        """
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer, 
                                   batch_size=16, 
                                   device="cuda" if torch.cuda.is_available() else "cpu", 
                                   column_text="dialogue", 
                                   column_summary="summary"):
        
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        logger.info(f"Evaluating {len(article_batches)} batches with a size of {batch_size} samples each...")
        for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):
            
            inputs = tokenizer(article_batch, max_length=1024, truncation=True, 
                               padding="max_length", return_tensors="pt")
            
            # Use beam search decoding parameters mapped from your configurations
            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device), 
                length_penalty=0.8, 
                num_beams=8, 
                max_length=128
            )
            
            # Decode the predicted numerical token matrix arrays back into human text
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, 
                                                 clean_up_tokenization_spaces=True) 
                                 for s in summaries]      
            
            decoded_summaries = [d.strip() for d in decoded_summaries]
            
            metric.add_batch(predictions=decoded_summaries, references=target_batch)
            
        # Compute and aggregate precision scores
        score = metric.compute()
        return score

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"Model Evaluation stage initiated. Target environment device: {device}")
        
        logger.info(f"Loading local tokenizer configuration from: {self.config.tokenizer_path}")
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        
        logger.info(f"Loading pre-trained Colab model weights from: {self.config.model_path}")
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
       
        logger.info(f"Reading tokenized evaluation datasets from disk target: {self.config.data_path}")
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # Initialize the target evaluation metrics layout matrix
        rouge_metric = evaluate.load('rouge')

        logger.info("Running benchmark predictions over the first 10 testing slices...")
        # Grabbing the slice [0:10] as specified to prevent CPU timeouts
        test_dataset_slice = dataset_samsum_pt['test'].select(range(10))

        score = self.calculate_metric_on_test_ds(
            dataset=test_dataset_slice, 
            metric=rouge_metric, 
            model=model_pegasus, 
            tokenizer=tokenizer, 
            batch_size=2, 
            column_text='dialogue', 
            column_summary='summary',
            device=device
        )

        # Extract precise metrics. The evaluate library returns raw float scores directly.
        rouge_dict = {
            "rouge1": score["rouge1"],
            "rouge2": score["rouge2"],
            "rougeL": score["rougeL"],
            "rougeLsum": score["rougeLsum"]
        }

        # Structure metric matrix and save output
        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        logger.info(f"Saving computed evaluation score matrix to csv: {self.config.metric_file_name}")
        df.to_csv(self.config.metric_file_name, index=False)
        logger.info("Model evaluation pipeline phase successfully completed!")

In [14]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.evaluate()
except Exception as e: 
    raise e

[2026-06-28 17:09:47,892: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 17:09:47,990: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 17:09:47,999: INFO: common: Created directory at: artifacts]
[2026-06-28 17:09:48,007: INFO: common: Created directory at: artifacts\model_evaluation]
[2026-06-28 17:09:48,007: INFO: 3303038261: Model Evaluation stage initiated. Target environment device: cpu]
[2026-06-28 17:09:48,012: INFO: 3303038261: Loading local tokenizer configuration from: artifacts\model_trainer\tokenizer]
[2026-06-28 17:09:50,867: INFO: 3303038261: Loading pre-trained Colab model weights from: artifacts\model_trainer\pegasus-samsum-model]


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 3736.72it/s]


[2026-06-28 17:09:55,722: INFO: 3303038261: Reading tokenized evaluation datasets from disk target: artifacts\data_transformation\samsum_dataset]
[2026-06-28 17:09:58,090: INFO: 3303038261: Running benchmark predictions over the first 10 testing slices...]
[2026-06-28 17:09:58,097: INFO: 3303038261: Evaluating 5 batches with a size of 2 samples each...]


100%|██████████| 5/5 [05:52<00:00, 70.54s/it]

[2026-06-28 17:15:50,869: INFO: rouge_scorer: Using default tokenizer.]


[2026-06-28 17:15:51,205: INFO: 3303038261: Saving computed evaluation score matrix to csv: artifacts\model_evaluation\metrics.csv]
[2026-06-28 17:15:51,224: INFO: 3303038261: Model evaluation pipeline phase successfully completed!]
